In [45]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [46]:
# 1. Prepare the data frame
cols = ["Layer", "E/I", "neuron-type", "%"]
data = np.array([['5', 'Excitatory', 'PC', 56.4],
        ['2/3', 'Excitatory', 'PC', 11.5],
        ['5', 'Inhibitory', 'FS', 85],
        ['5', 'Inhibitory', 'non-FS', 0],
        ['5', 'Inhibitory', 'FS', 27.9],
        ['5', 'Inhibitory', 'FS', 80],
        ['5', 'Inhibitory', 'FS', 74],
        ['2/3', 'Inhibitory', 'FS', 69.4],
        ['2/3', 'Inhibitory', 'non-FS', 7.1],
        ['5', 'Inhibitory', 'FS', 90]])
data1 = pd.DataFrame(data = data, columns = cols)

data_hippo = np.array([['CA1-3', 'Excitatory', 'PC', 0.],
                       ['Sub', 'Excitatory', 'PC', 51.6],
                       ['CA1-3', 'Excitatory', 'PC', 0.],
                       ['CA1-3', 'Inhibitory', 'basket', 100.],
                       ['CA1-3', 'Inhibitory', 'bistratified', 65.],
                       ['CA1-3', 'Inhibitory', 'axo-axonic', 0.],
                       ['CA1-3', 'Inhibitory', 'basket', 100.],
                       ['CA1-3', 'Inhibitory', 'bistratified', 100.],])

data1 = pd.DataFrame(data_hippo, columns = cols)

data1['%'] = pd.to_numeric(data1['%'])


In [47]:


# 2. Initialize side-by-side subplots for each layer (sharing the y-axis)
fig, axes = plt.subplots(2, 1, figsize=(5, 11) )

#layers = ['2/3', '5']
#neuron_order = ['PC', 'FS', 'non-FS'] # Consistent ordering for y-axis
layers = ['CA1-3', 'Sub']
neuron_order = ['PC', 'basket', 'bistratified', 'axo-axonic']

custom_palette = {'Excitatory': '#2678fc', 'Inhibitory': '#f20c4d'} # Distinct colors for E vs I

for i, layer in enumerate(layers):
    # Filter data for the specific layer
    df_layer = data1[data1['Layer'] == layer]
    
    # Generate the horizontal bar plot
    sns.barplot(
        data=df_layer,
        x='%',                  # Swapped x and y to change orientation to horizontal
        y='neuron-type',
        hue='E/I',
        order=neuron_order,
        hue_order=['Excitatory', 'Inhibitory'],
        ax=axes[i],
        errorbar='sd',          # Standard deviation error bars
        capsize=0.1,
        palette=custom_palette,
        dodge=False             # Centers the bars cleanly
    )
    
    # Titles and Labels
    axes[i].set_title(f'Layer {layer}', fontsize=13, pad=10)
    axes[i].set_xlabel('$\\%$ of Neurons with autapses', fontsize=11, labelpad=8)
    axes[i].set_xlim((0., 100.))
    # Remove borders (spines) from individual axes
    sns.despine(ax=axes[i], top=True, right=True, left=True, bottom=True)
    
    # Clean up duplicate legends and y-axis labels
    if i == 0:
        axes[i].set_ylabel('Neuron Type', fontsize=11, labelpad=8)
        axes[i].get_legend().remove()
    else:
        axes[i].set_ylabel('')
        axes[i].legend(loc='upper right') # Moved to lower right to avoid bar collision

# 3. Final layout adjustments and saving
plt.suptitle('Proportion ($\\%$) of Neurons Forming Autapses', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('neuron_type_by_ei_faceted_horizontal_HIPPO.png', dpi=300, bbox_inches='tight')
plt.close()

In [52]:

# Calculate the mean percentage for groups with multiple observations (like Layer 5 FS)
df_mean = data1.groupby(['Layer', 'neuron-type', 'E/I'])['%'].mean().reset_index()

# 2. Initialize a 2x3 grid of subplots (2 Layers x 3 Neuron Types)
fig, axes = plt.subplots(2, 4, figsize=(12, 8))

#layers = ['2/3', '5']
#neuron_order = ['PC', 'FS', 'non-FS']
layers = ['CA1-3', 'Sub']
neuron_order = ['PC', 'basket', 'bistratified', 'axo-axonic']

custom_palette = {'Excitatory': '#2678fc', 'Inhibitory': '#f20c4d'}
remainder_color = '#e5e5e5' # Light gray for neurons without autapses

# 3. Populate each subplot with a pie chart
for row, layer in enumerate(layers):
    for col, neuron in enumerate(neuron_order):
        ax = axes[row, col]
        
        # Extract the specific row matching current Layer and Neuron Type
        match = df_mean[(df_mean['Layer'] == layer) & (df_mean['neuron-type'] == neuron)]
        
        if not match.empty:
            with_autapses = match['%'].values[0]
            ei_status = match['E/I'].values[0]
            primary_color = custom_palette[ei_status]
        else:
            with_autapses = 0.0
            primary_color = remainder_color
            
        without_autapses = 100.0 - with_autapses
        
        # Slices represent the whole (With vs. Without)
        slices = [with_autapses, without_autapses]
        colors = [primary_color, remainder_color]
        
        # Build the pie chart
        wedges, texts, autotexts = ax.pie(
            slices, 
            startangle=90, 
            colors=colors,
            autopct=lambda p: f'{p:.1f}%' if p > 0 else '', # Hide label if 0%
            pctdistance=0.55,
            wedgeprops=dict(width=0.4, edgecolor='w') # Donut-style cutout for clean look
        )
        
        # Style the percentage text labels inside the slices
        for at in autotexts:
            at.set_fontsize(18)
            at.set_weight('bold')
            # If a slice is 100% gray (like 0% autapses), make the text readable against gray
            at.set_color('black')


        # Add titles to individual subplots
        ax.set_title(f'Layer {layer}: {neuron}', fontsize=12, pad=8)

# 4. Global title and layout configuration
plt.suptitle('Proportion ($\\%$) of Neurons Forming Autapses', fontsize=15, y=0.98)
plt.tight_layout()
plt.savefig('neuron_type_pie_charts_HIPPO.png', dpi=300, bbox_inches='tight')
plt.close()

In [56]:
# Calculate the mean percentage for groups with multiple observations (like Layer 5 FS)
df_mean = data1.groupby(['Layer', 'neuron-type', 'E/I'])['%'].mean().reset_index()

# 2. Initialize a 2x4 grid of subplots (2 Layers x 4 Neuron Types)
fig, axes = plt.subplots(2, 4, figsize=(12, 8))

layers = ['CA1-3', 'Sub']
neuron_order = ['PC', 'basket', 'bistratified', 'axo-axonic']

custom_palette = {'Excitatory': '#2678fc', 'Inhibitory': '#f20c4d'}
remainder_color = '#e5e5e5' # Light gray for neurons without autapses

# 3. Populate each subplot with a pie chart
for row, layer in enumerate(layers):
    for col, neuron in enumerate(neuron_order):
        ax = axes[row, col]
        
        # Extract the specific row matching current Layer and Neuron Type
        match = df_mean[(df_mean['Layer'] == layer) & (df_mean['neuron-type'] == neuron)]
        
        if not match.empty:
            with_autapses = match['%'].values[0]
            ei_status = match['E/I'].values[0]
            primary_color = custom_palette[ei_status]
        else:
            with_autapses = 0.0
            primary_color = remainder_color
            
        without_autapses = 100.0 - with_autapses
        
        # Slices represent the whole (With vs. Without)
        slices = [with_autapses, without_autapses]
        colors = [primary_color, remainder_color]
        
        # Build the pie chart
        wedges, texts, autotexts = ax.pie(
            slices, 
            startangle=90, 
            colors=colors,
            autopct=lambda p: f'{p:.1f}%', # Hides label if value is 0%
            pctdistance=0.78,                             # Moves text outwards into the middle of the wedge ring
            wedgeprops=dict(width=0.4, edgecolor='w')      # Donut-style cutout
        )
        
        # --- POSITION AND STYLE TEXT IN THE WEDGE ---
        # Hide the text for the second slice (the gray remainder)
        if len(autotexts) > 1:
            autotexts[1].set_text('')
            
        # Style the autapse proportion label inside its colored slice
        autotexts[0].set_fontsize(13) # Reduced slightly from 18 to fit nicely inside the wedge boundary
        autotexts[0].set_weight('bold')
        
        # Make the text highly visible against the colored backgrounds
        autotexts[0].set_color('black')


        # Add titles to individual subplots
        ax.set_title(f'Layer {layer}: {neuron}', fontsize=12, pad=8)

# 4. Global title and layout configuration
plt.suptitle('Proportion ($\\%$) of Neurons Forming Autapses', fontsize=15, y=0.98)
plt.tight_layout()
plt.savefig('neuron_type_pie_charts_HIPPO.png', dpi=300, bbox_inches='tight')
plt.close()

In [49]:
cols = ['connection', 'proportion']
data2 = np.array([['PFC -> Hb', 84.8],
                  ['PFC -> Pons', 59.3],
                  ['PFC -> cPFC', 2.6]])
data_hippo = np.array([['Sub -> NAc', 46.6],
                       ['Sub -> Amygdala', 17.2]])

df2 = pd.DataFrame(data = data_hippo, columns = cols)
df2['proportion'] = pd.to_numeric(df2['proportion'])
print(df2.head())

        connection  proportion
0       Sub -> NAc        46.6
1  Sub -> Amygdala        17.2


In [51]:
# 2. Initialize subplots stacked vertically in 3 rows and 1 column
fig, axes = plt.subplots(2, 1, figsize=(5, 9))

# Professional color palette (Blue for the connection proportion, Light Gray for the remainder)
primary_color = '#2678fc'
remainder_color = '#e5e5e5'

for i, row in df2.iterrows():
    ax = axes[i]
    prop = row['proportion']
    conn = row['connection']
    
    # Slices represent the connection proportion vs the remainder
    slices = [prop, 100 - prop]
    
    # Generate the donut chart structure
    wedges, texts = ax.pie(
        slices,
        startangle=90,
        colors=[primary_color, remainder_color],
        wedgeprops=dict(width=0.4, edgecolor='w') 
    )
    
    # --- PLACE TEXT IN THE CENTER ---
    # Placing text at coordinates (0, 0) centers it perfectly inside the donut hole
    ax.text(
        0, 0, 
        f'{prop:.1f}%', 
        ha='center', 
        va='center', 
        fontsize=12, 
        fontweight='bold', 
        color='black'
    )
        
    # Set the connection label as the horizontal y-axis label on the left side
    ax.set_ylabel(conn, rotation=0, labelpad=60, va='center', fontsize=12, fontweight='bold')

# 3. Add a global title and save the figure
plt.suptitle('Proportion ($\\%$) by Connection Type', fontsize=14, y=0.98, fontweight='bold')
plt.tight_layout()
plt.savefig('connection_pie_charts_HIPPO.png', dpi=300, bbox_inches='tight')
plt.close()